In [ ]:
import cv2
from matplotlib import pyplot
import numpy

# Ustawienie rozmarów wyświetlanych obrazów
pyplot.rcParams["figure.figsize"] = (18, 10)

In [ ]:
#
# Wczytanie obrazów źródłowych
#
# źródło grafiki colors.jpg: https://unsplash.com/photos/gT5kuls6Y6Q
# źródło grafiki gacław-na-kuchni.jpg: własne
#
#image_from_file = cv2.imread('images/colors.jpg')
image_from_file = cv2.imread('images/gacław-na-kuchni.jpg')
image_gray = cv2.cvtColor(image_from_file, cv2.COLOR_BGR2GRAY)
image_color = cv2.cvtColor(image_from_file, cv2.COLOR_BGR2RGB)
print('Rozmiar obrazka: ', image_from_file.shape)

# Zadanie 1

In [ ]:
output = numpy.copy(image_gray).astype(numpy.float32)

def find_closest_palette_color(value):
    return round(value / 255) * 255

#
# Algorytm
#
for y in range(output.shape[0]):
    for x in range(output.shape[1]):
        oldpixel = output[y][x]
        newpixel = find_closest_palette_color(oldpixel)
        output[y][x] = newpixel
        quant_error = oldpixel - newpixel

        if x + 1 < output.shape[1]:
            output[y][x + 1] += quant_error * 7 / 16
        if y + 1 < output.shape[0]:
            if x > 0:
                output[y + 1][x - 1] += quant_error * 3 / 16
            output[y + 1][x] += quant_error * 5 / 16
            if x + 1 < output.shape[1]:
                output[y + 1][x + 1] += quant_error * 1 / 16

# Przycięcie wartości, żeby nie wyjść poza zakres uint8
output = numpy.clip(output, 0, 255).astype(numpy.uint8)

#
# Wyświetlenie
#
pyplot.imshow(output, cmap='gray')
pyplot.axis('off')

In [ ]:
#
# Histogram
#
histr = cv2.calcHist([output], [0], None, [256], [0, 256])
pyplot.plot(histr)
pyplot.xlim([-1, 256])
pyplot.xlabel('Wartośc składowej koloru []')
pyplot.ylabel('Liczba pikseli obrazu []')

# Zadanie 2

In [ ]:
# Parametr: liczba poziomów jasności na składową (domyślnie 2: 0 i 255)
k = 9

# Redukcja kolorów do palety (bez ditheringu)
reduced = numpy.copy(image_color).astype(numpy.float32)

def quantize(value, k):
    return round((k - 1) * value / 255) * 255 / (k - 1)

for y in range(reduced.shape[0]):
    for x in range(reduced.shape[1]):
        for c in range(3):  # R, G, B
            reduced[y, x, c] = quantize(reduced[y, x, c], k)


output = numpy.copy(image_color).astype(numpy.float32)

#
# Algorytm
#
for y in range(output.shape[0]):
    for x in range(output.shape[1]):
        oldpixel = output[y, x].copy()
        newpixel = numpy.array([quantize(c, k) for c in oldpixel])
        output[y, x] = newpixel
        quant_error = oldpixel - newpixel

        if x + 1 < output.shape[1]:
            output[y, x + 1] += quant_error * 7 / 16
        if y + 1 < output.shape[0]:
            if x > 0:
                output[y + 1, x - 1] += quant_error * 3 / 16
            output[y + 1, x] += quant_error * 5 / 16
            if x + 1 < output.shape[1]:
                output[y + 1, x + 1] += quant_error * 1 / 16


output = numpy.clip(output, 0, 255).astype(numpy.uint8)
reduced = numpy.clip(reduced, 0, 255).astype(numpy.uint8)


#
# Wyświetlenie
#
pyplot.figure(figsize=(12, 16))

pyplot.subplot(2, 1, 1)
pyplot.title('Obrazek po samej redukcji barw')
pyplot.imshow(reduced)

pyplot.subplot(2, 1, 2)
pyplot.title('Obrazek w kolorze po algorytmie Floyda-Steinberga')
pyplot.imshow(output)

In [ ]:
#
# Histogram
#
color = ('r', 'g', 'b')

for i, col in enumerate(color):
    histr = cv2.calcHist([output], [i], None, [256], [0, 256])
    pyplot.plot(histr, color=col)
    pyplot.xlim([-1, 256])
    pyplot.xlabel('Wartośc składowej koloru []')
    pyplot.ylabel('Liczba pikseli obrazu []')

# Zadanie 3

In [ ]:
#
# Przygotowanie płótna
#
width = 80
height = 60
image = numpy.zeros((height, width, 3), dtype=numpy.uint8)


#
# Funkcja rysująca punkt
#
# NOTE(sdatko): punkt 0,0 to lewy dolny róg obrazu
#
def draw_point(image, x, y, color=(255, 255, 255)):
    if 0 <= x < image.shape[1] and 0 <= y < image.shape[0]:
        image[image.shape[0] - 1 - y, x, :] = color


#
# Funkcja rysująca linię
#
def draw_line(image, x1, y1, x2, y2, color=(255, 255, 255)):
    dx = abs(x2 - x1)
    dy = abs(y2 - y1)
    sx = 1 if x1 < x2 else -1
    sy = 1 if y1 < y2 else -1
    err = dx - dy

    while True:
        draw_point(image, x1, y1, color)
        if x1 == x2 and y1 == y2:
            break
        e2 = 2 * err
        if e2 > -dy:
            err -= dy
            x1 += sx
        if e2 < dx:
            err += dx
            y1 += sy

# Checking if a point belongs to a triangle
def is_point_in_triangle(p, a, b, c):
    def sign(p1, p2, p3):
        return (p3[0] - p1[0]) * (p2[1] - p1[1]) - (p3[1] - p1[1]) * (p2[0] - p1[0])
    
    d1 = sign(a, b, p)
    d2 = sign(b, c, p)
    d3 = sign(c, a, p)

    has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
    has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)

    return not (has_neg and has_pos)

#
# Funkcja rysująca trójkąt
#
def draw_triangle(image, a, b, c, color=(255, 255, 255)):
    # Bounding box
    xmin = max(min(a[0], b[0], c[0]), 0)
    xmax = min(max(a[0], b[0], c[0]), image.shape[1] - 1)
    ymin = max(min(a[1], b[1], c[1]), 0)
    ymax = min(max(a[1], b[1], c[1]), image.shape[0] - 1)

    for y in range(ymin, ymax + 1):
        for x in range(xmin, xmax + 1):
            if is_point_in_triangle((x, y), a, b, c):
                draw_point(image, x, y, color)


#
# Rysowanie
#
draw_line(image, 10, 10, 30, 50, color=(255, 0, 0))  # czerwona linia
draw_triangle(image, (20, 10), (40, 40), (60, 15), color=(0, 255, 0))  # zielony trójkąt

#
# Wyświetlenie
#
pyplot.imshow(image)

# Zadanie 4

In [ ]:
#
# Przygotowanie płótna
#
width = 80
height = 60
image = numpy.zeros((height, width, 3), dtype=numpy.uint8)


#
# Funkcja rysująca punkt
#
# NOTE(sdatko): punkt 0,0 to lewy dolny róg obrazu
#
def draw_point(image, x, y, color=(255, 255, 255)):
    if 0 <= x < image.shape[1] and 0 <= y < image.shape[0]:
        image[image.shape[0] - 1 - y, x, :] = color


#
# Funkcja rysująca linię
#
def draw_line(image, x1, y1, x2, y2, color1, color2):
    dx = abs(x2 - x1)
    dy = abs(y2 - y1)
    sx = 1 if x1 < x2 else -1
    sy = 1 if y1 < y2 else -1
    err = dx - dy

    length = max(dx, dy) if max(dx, dy) != 0 else 1
    i = 0

    while True:
        t = i / length
        color = numpy.array(color1) +  t * (numpy.array(color2) - numpy.array(color1))
        draw_point(image, x1, y1, color.astype(numpy.uint8))

        if x1 == x2 and y1 == y2:
            break
        e2 = 2 * err
        if e2 > -dy:
            err -= dy
            x1 += sx
        if e2 < dx:
            err += dx
            y1 += sy
        i += 1


def triangle_area(p1, p2, p3):
    return abs((p1[0] * (p2[1] - p3[1]) +
                p2[0] * (p3[1] - p1[1]) +
                p3[0] * (p1[1] - p2[1])) / 2.0)

#
# Funkcja rysująca trójkąt
#
def draw_triangle(image, a, b, c, color_a, color_b, color_c):
    xmin = max(min(a[0], b[0], c[0]), 0)
    xmax = min(max(a[0], b[0], c[0]), image.shape[1] - 1)
    ymin = max(min(a[1], b[1], c[1]), 0)
    ymax = min(max(a[1], b[1], c[1]), image.shape[0] - 1)

    total_area = triangle_area(a, b, c)

    for y in range(ymin, ymax + 1):
        for x in range(xmin, xmax + 1):
            p = (x, y)
            a0 = triangle_area(p, b, c)
            a1 = triangle_area(a, p, c)
            a2 = triangle_area(a, b, p)
            w0 = a0 / total_area
            w1 = a1 / total_area
            w2 = a2 / total_area

            if (w0 >= 0) and (w1 >= 0) and (w2 >= 0) and abs(w0 + w1 + w2 - 1) < 0.01:
                color = w0 * numpy.array(color_a) + w1 * numpy.array(color_b) + w2 * numpy.array(color_c)
                draw_point(image, x, y, color.astype(numpy.uint8))


#
# Rysowanie
#
draw_line(image, 10, 10, 30, 50, (255, 0, 0), (0, 0, 255))
draw_triangle(image, (20, 10), (50, 50), (70, 20),
              (255, 0, 0), (0, 255, 0), (0, 0, 255))

#
# Wyświetlenie
#
pyplot.imshow(image)

# Zadanie 5

In [ ]:
#
# Przygotowanie płótna
#
scale = 2

width = 1080
height = 720

supersampled_width = width * scale
supersampled_height = height * scale

image = numpy.zeros((height, width, 3), dtype=numpy.uint8)
super_image = numpy.zeros((supersampled_height, supersampled_width, 3), dtype=numpy.uint8)


#
# Rysowanie
#
draw_line(super_image, 10 * scale, 10 * scale, 30 * scale, 50 * scale, (255, 0, 0), (0, 0, 255))
draw_triangle(super_image, (200 * scale, 100 * scale), (500 * scale, 500 * scale),
              (700 * scale, 200 * scale), (255, 0, 0), (0, 255, 0), (0, 0, 255))


for y in range(height):
    for x in range(width):
        block = super_image[  # blok rozmiaru scale x scale (2 x 2)
            y * scale:y * scale + scale, 
            x * scale:x * scale + scale, 
            :
        ]
        # usrednienie 4 pikseli do pikselu obrazu wynikowego
        image[y, x, :] = numpy.mean(block, axis=(0, 1)).astype(numpy.uint8)
        
#
# Wyświetlenie
#
pyplot.imshow(image)